
# Fase 1 — Análisis exploratorio de datos

## Bike Sharing Analytics

Universidad de La Salle  
Maestría en Inteligencia Artificial  
Proyecto Integrador de Ciencia de Datos

### Naturaleza del notebook

Reconstrucción independiente del análisis exploratorio
integrado en el dashboard global.

No corresponde al notebook académico original de Fase 1.

### Formulación del problema

Identificar los patrones temporales, meteorológicos
y de comportamiento de los usuarios asociados con
la demanda diaria del sistema de bicicletas compartidas.

### Objetivo general

Analizar los factores asociados con la demanda de
bicicletas compartidas durante 2011 y 2012.

### Objetivos específicos

- Caracterizar la calidad y estructura del dataset.
- Identificar patrones temporales de demanda.
- Analizar la relación entre clima y alquileres.
- Identificar valores atípicos mediante IQR y Z-score.
- Generar evidencia descriptiva para las fases de modelado.

### Fuente de datos

UCI Machine Learning Repository — Bike Sharing Dataset.

DOI: 10.24432/C5W894

https://archive.ics.uci.edu/dataset/275/bike-sharing-dataset


In [ ]:

# Localizar la raíz del repositorio.

from pathlib import Path
import sys
import hashlib

actual = Path.cwd().resolve()

candidatos = [actual, *actual.parents]

# Alternativa para Google Colab.

candidatos.append(
    Path("/content/Bike-Sharing-Analytics")
)

ROOT = next(
    (
        p for p in candidatos
        if (p / "data" / "day.csv").is_file()
        and (p / "requirements.txt").is_file()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Descarga o clona el repositorio antes de ejecutar."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import DATASET, RESULTS_DIR

HASH_ESPERADO = (
    "a6bcf826782d3c0fbfdcbeead17cd0884"
    "185a0dafe8ff10cd48a874ee7ba18be"
)

assert hashlib.sha256(
    DATASET.read_bytes()
).hexdigest() == HASH_ESPERADO

print("Repositorio:", ROOT)
print("Dataset:", DATASET)
print("Integridad del dataset: OK")


## 1. Carga y calidad de los datos

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency

SALIDA = RESULTS_DIR / "reproduccion" / "fase1"

SALIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATASET)

df["dteday"] = pd.to_datetime(df["dteday"])

df = df.sort_values("dteday").reset_index(drop=True)

calidad = pd.DataFrame({
    "indicador": [
        "Registros",
        "Variables",
        "Valores faltantes",
        "Filas duplicadas",
        "Fechas duplicadas",
        "Inconsistencias en demanda"
    ],
    "valor": [
        len(df),
        df.shape[1],
        df.isna().sum().sum(),
        df.duplicated().sum(),
        df["dteday"].duplicated().sum(),
        (df["cnt"] !=
         df["casual"] + df["registered"]).sum()
    ]
})

display(calidad)

assert df.shape == (731, 16)

calidad.to_csv(
    SALIDA / "calidad_datos.csv",
    index=False
)

display(df.head())



## 2. Caracterización y estadística descriptiva

La variable objetivo es cnt, correspondiente al
total diario de alquileres.

Las variables casual y registered son componentes
directos de cnt. Por tanto, no deben utilizarse como
predictores de la demanda total en los modelos.


In [ ]:

diccionario = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "valores_unicos": df.nunique().values,
    "faltantes": df.isna().sum().values
})

display(diccionario)

diccionario.to_csv(
    SALIDA / "diccionario_estructura.csv",
    index=False
)

descriptivos = df.describe().T

display(descriptivos)

descriptivos.to_csv(
    SALIDA / "estadistica_descriptiva.csv"
)


## 3. Indicadores globales y temporales

In [ ]:

total = int(df["cnt"].sum())

indicadores = {
    "registros": len(df),
    "alquileres_totales": total,
    "promedio_diario": float(df["cnt"].mean()),
    "mediana_diaria": float(df["cnt"].median()),
    "demanda_maxima": int(df["cnt"].max()),
    "demanda_minima": int(df["cnt"].min()),
    "alquileres_casuales": int(df["casual"].sum()),
    "alquileres_registrados": int(df["registered"].sum()),
    "participacion_registrados_pct":
        float(df["registered"].sum() / total * 100)
}

df_indicadores = pd.DataFrame(
    indicadores.items(),
    columns=["indicador", "valor"]
)

display(df_indicadores)

df_indicadores.to_csv(
    SALIDA / "indicadores_globales.csv",
    index=False
)

anual = df.groupby(
    df["dteday"].dt.year
)[["cnt", "casual", "registered"]].sum()

mensual = df.groupby(
    df["dteday"].dt.to_period("M")
)[["cnt", "casual", "registered"]].sum()

display(anual)
display(mensual)

anual.to_csv(SALIDA / "demanda_anual.csv")
mensual.to_csv(SALIDA / "demanda_mensual.csv")

crecimiento = (
    anual.loc[2012, "cnt"] /
    anual.loc[2011, "cnt"] - 1
) * 100

print("Crecimiento anual (%):", round(crecimiento, 4))



## 4. Análisis de correlaciones

Se utilizan Pearson y Spearman para estudiar
asociaciones lineales y monotónicas.

La correlación no implica causalidad.


In [ ]:

variables = [
    "temp", "atemp", "hum", "windspeed",
    "casual", "registered", "cnt"
]

pearson = df[variables].corr(method="pearson")
spearman = df[variables].corr(method="spearman")

display(pearson.round(4))
display(spearman.round(4))

pearson.to_csv(SALIDA / "correlacion_pearson.csv")
spearman.to_csv(SALIDA / "correlacion_spearman.csv")

for nombre, matriz in [
    ("pearson", pearson),
    ("spearman", spearman)
]:

    plt.figure(figsize=(9, 7))

    sns.heatmap(
        matriz,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1
    )

    plt.title("Correlación de " + nombre.capitalize())
    plt.tight_layout()

    plt.savefig(
        SALIDA / f"correlacion_{nombre}.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()



## 5. Detección de valores atípicos

Se aplica el rango intercuartílico con factor 1.5
y Z-score con umbral absoluto de 3.

La detección es exploratoria. No se eliminan
registros ni se modifica el dataset.


In [ ]:

registros = []

for variable in variables:

    serie = df[variable]

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)

    iqr = q3 - q1

    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr

    mascara_iqr = (
        (serie < inferior) |
        (serie > superior)
    )

    z = (
        serie - serie.mean()
    ) / serie.std(ddof=0)

    registros.append({
        "variable": variable,
        "limite_inferior_iqr": inferior,
        "limite_superior_iqr": superior,
        "outliers_iqr": int(mascara_iqr.sum()),
        "outliers_zscore": int((z.abs() > 3).sum())
    })

outliers = pd.DataFrame(registros)

display(outliers)

outliers.to_csv(
    SALIDA / "deteccion_outliers.csv",
    index=False
)

print(
    "Detecciones IQR:",
    outliers["outliers_iqr"].sum()
)

print(
    "Detecciones Z-score:",
    outliers["outliers_zscore"].sum()
)

outliers.set_index("variable")[
    ["outliers_iqr", "outliers_zscore"]
].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Comparación de métodos de detección")
plt.ylabel("Detecciones")
plt.tight_layout()

plt.savefig(
    SALIDA / "comparacion_outliers.png",
    dpi=150
)

plt.show()
plt.close()


## 6. Visualizaciones de demanda

In [ ]:

estaciones = {
    1: "Invierno",
    2: "Primavera",
    3: "Verano",
    4: "Otoño"
}

climas = {
    1: "Despejado",
    2: "Niebla o nublado",
    3: "Lluvia o nieve ligera",
    4: "Clima severo"
}

df_eda = df.assign(
    estacion=df["season"].map(estaciones),
    clima=df["weathersit"].map(climas)
)

demanda_clima = df_eda.groupby("clima")["cnt"].agg(
    ["count", "mean", "median", "sum"]
)

demanda_estacion = df_eda.groupby("estacion")["cnt"].agg(
    ["count", "mean", "median", "sum"]
)

display(demanda_clima)
display(demanda_estacion)

demanda_clima.to_csv(SALIDA / "demanda_clima.csv")
demanda_estacion.to_csv(SALIDA / "demanda_estacion.csv")

graficas = [
    (
        "histograma_demanda",
        lambda: sns.histplot(df["cnt"], kde=True),
        "Distribución de la demanda diaria"
    ),
    (
        "demanda_diaria",
        lambda: plt.plot(df["dteday"], df["cnt"]),
        "Evolución temporal de la demanda"
    ),
    (
        "temperatura_demanda",
        lambda: sns.scatterplot(
            data=df, x="temp", y="cnt"
        ),
        "Temperatura y demanda"
    ),
    (
        "demanda_clima",
        lambda: sns.boxplot(
            data=df_eda, x="clima", y="cnt"
        ),
        "Demanda según condiciones climáticas"
    ),
    (
        "demanda_estacion",
        lambda: sns.boxplot(
            data=df_eda, x="estacion", y="cnt"
        ),
        "Demanda según estación"
    )
]

for nombre, funcion, titulo in graficas:

    plt.figure(figsize=(11, 5))

    funcion()

    plt.title(titulo)
    plt.xticks(rotation=15)
    plt.tight_layout()

    plt.savefig(
        SALIDA / f"{nombre}.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()



## 7. Análisis de asociación categórica

Se utiliza la prueba Chi-cuadrado y V de Cramér.

Se aplica corrección de Bonferroni para controlar
las comparaciones múltiples.

La dependencia temporal entre observaciones limita
la interpretación inferencial de estas pruebas.


In [ ]:

pares = [
    ("season", "weathersit"),
    ("weekday", "weathersit"),
    ("workingday", "weathersit"),
    ("holiday", "weathersit")
]

resultados_chi = []

for variable1, variable2 in pares:

    tabla = pd.crosstab(
        df[variable1],
        df[variable2]
    )

    chi2, p, gl, esperadas = chi2_contingency(
        tabla,
        correction=False
    )

    n = tabla.to_numpy().sum()

    denominador = n * min(
        tabla.shape[0] - 1,
        tabla.shape[1] - 1
    )

    cramer = np.sqrt(
        chi2 / denominador
    )

    resultados_chi.append({
        "variable_1": variable1,
        "variable_2": variable2,
        "chi2": chi2,
        "p_valor": p,
        "p_ajustado": min(p * len(pares), 1),
        "grados_libertad": gl,
        "v_cramer": cramer,
        "esperadas_menores_5_pct":
            float((esperadas < 5).mean() * 100)
    })

df_chi = pd.DataFrame(resultados_chi)

display(df_chi)

df_chi.to_csv(
    SALIDA / "pruebas_chi_cuadrado.csv",
    index=False
)



## 8. Conclusiones

Los resultados descriptivos permiten identificar
patrones temporales y meteorológicos asociados
con la demanda.

Las correlaciones entre temperatura y sensación
térmica deben considerarse durante la selección
de características.

La detección de valores atípicos no implica
automáticamente su eliminación.

Las variables casual y registered deben excluirse
como predictores de cnt para prevenir fuga de datos.

El análisis exploratorio proporciona la base para
las siguientes fases de regresión, modelado avanzado
y preprocesamiento.


In [ ]:

import json

resumen_fase1 = {
    "fase": 1,
    "tipo": "Reconstrucción independiente",
    "filas": int(len(df)),
    "columnas": int(df.shape[1]),
    "alquileres_totales": int(df["cnt"].sum()),
    "crecimiento_anual_pct": float(crecimiento),
    "detecciones_iqr": int(outliers["outliers_iqr"].sum()),
    "detecciones_zscore": int(
        outliers["outliers_zscore"].sum()
    ),
    "registros_eliminados": 0,
    "modelos_entrenados": 0
}

(SALIDA / "resumen_fase1.json").write_text(
    json.dumps(
        resumen_fase1,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== RESULTADOS FASE 1 ===")

for clave, valor in resumen_fase1.items():
    print(f"{clave}: {valor}")

print("\nEvidencias generadas:")

for archivo in sorted(SALIDA.iterdir()):
    print(" -", archivo.name)
